# S6E9 · V5：参考 heuljax 的方向加入 10-fold XGBoost + Rank Blend

你当前这份 notebook 已完成：

- 旧 CatBoost 5-fold OOF AUC：`0.9456182325`
- 新 LightGBM 5-fold OOF AUC：`0.9459463810`
- CatBoost/LightGBM 50/50 OOF AUC：`0.94598624`
- 你最新反馈的 Kaggle Public Score：**0.94618**

这版 V5 不再继续只微调 LightGBM，而是加入一个更值得尝试的新模型族：**10-fold XGBoost**。

原因：Kaggle 用户 **heuljax / Paul Bryan Elefante** 在 S6E9 的公开讨论中表示，他当时最好的单模型是 **10-fold XGBoost（CV 0.94652）**，Logistic Regression 约 **0.94640**。公开页面没有暴露他完整的私有参数，因此这里**不声称复制他的模型**；我们只复用公开可验证的核心方向：10-fold、XGBoost、离散/精确值信号、OOF 验证和模型融合。

同时参考本比赛公开 XGBoost 方案中反复出现的有效思路：
- digit decomposition
- frequency encoding
- leak-free target encoding
- 10-fold CV
- rank averaging / rank blending

本 notebook 保留你已经跑完的 CatBoost + LightGBM 结果，新训练只针对 XGBoost。不会自动向 Kaggle 提交。

参考：
- heuljax 的公开讨论：https://www.kaggle.com/competitions/playground-series-s6e9/discussion/742317
- XGBoost + Triple TE 公开 notebook：https://www.kaggle.com/code/najiama/xgboost-triple-te-dynamic-pruning-lb-0-94639


## 1. 加载原有训练与编码函数

下面保留你 V3 的算法：原来的行内数字特征、LightGBM 内层五折目标编码、相同外层五折和参数；仅把新训练目录前缀改为 `v4_lightgbm_`。函数定义不会开始训练。

目标编码仍严格在外层训练折内部计算；验证/测试只使用外层训练折的映射。不要为了省时间在整个训练集上先算一份目标编码。

原训练器保存模型、每折预测与编码来源的训练行索引；**模型文件本身不包含全部特征/目标编码过程**，以后推理必须配合相同代码与训练数据重建映射。

In [1]:
"""S6E9: fixed-fold CatBoost; optional LightGBM with cross-fitted target encoding.

Default: python s6e9_train_v3.py
Optional: python s6e9_train_v3.py --models catboost lightgbm
Dependencies: numpy pandas scikit-learn catboost; optional lightgbm.
No training occurs when this module is imported.
"""
import argparse
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "Will_Buy_EV"
NUMERIC = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
           "Environmental_Concern_Level"]


def features(frame, model_name):
    """Row-local features only: no labels or full-data target statistics."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()
    for col, scale, positions in [("Age", 1, 2), ("Annual_Income_USD", 1, 6),
                                 ("Daily_Commute_km", 10, 3),
                                 ("Charging_Stations_Near_Home", 1, 2),
                                 ("Charging_Stations_Near_Work", 1, 2)]:
        values = np.rint(frame[col].to_numpy() * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (values // 10**digit % 10).astype(np.int8)
    income = np.rint(frame["Annual_Income_USD"]).astype(np.int64)
    x["income_last_2"] = income % 100
    x["income_last_3"] = income % 1000
    if model_name == "catboost":
        for col in NUMERIC:
            scale = 10 if col == "Daily_Commute_km" else 1
            x[f"{col}_as_category"] = np.rint(frame[col] * scale).astype(np.int64).astype(str)
    else:
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000
    return x


def encode_fold(train_x, train_y, valid_x, test_x, seed, smoothing=1.0):
    """Inner OOF for training TE; outer-training-only maps for validation/test."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y)
    inner = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(a, labels))
    cats = a.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    keys = NUMERIC + cats + ["income_prefix_100", "income_prefix_1000"]
    for col in keys:
        encoded = np.full(len(a), np.nan)
        for fit_idx, hold_idx in inner:
            subset = pd.DataFrame({"key": a[col].iloc[fit_idx].to_numpy(), "y": labels[fit_idx]})
            stats = subset.groupby("key")["y"].agg(["sum", "count"])
            prior = labels[fit_idx].mean()
            mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
            encoded[hold_idx] = a[col].iloc[hold_idx].map(mapping).fillna(prior).to_numpy()
        stats = pd.DataFrame({"key": a[col], "y": labels}).groupby("key")["y"].agg(["sum", "count"])
        prior = labels.mean()
        mapping = (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)
        a[f"{col}_te"] = encoded
        b[f"{col}_te"] = b[col].map(mapping).fillna(prior).to_numpy()
        c[f"{col}_te"] = c[col].map(mapping).fillna(prior).to_numpy()
        assert np.isfinite(encoded).all()
    # Label-free frequency: exclude a training row's own count contribution.
    counts = a["Annual_Income_USD"].value_counts()
    a["income_frequency"] = (a["Annual_Income_USD"].map(counts) - 1) / (len(a) - 1)
    for frame in (b, c):
        frame["income_frequency"] = frame["Annual_Income_USD"].map(counts).fillna(0) / len(a)
    for col in cats:
        dtype = pd.CategoricalDtype(categories=pd.unique(a[col]))
        for frame in (a, b, c):
            frame[col] = frame[col].astype(dtype)
    return a, b, c


def run(project="/Users/yw/unsw/Kaggle", models=("catboost",), threads=-1):
    models = tuple(models)
    if not models or len(set(models)) != len(models) or not set(models) <= {"catboost", "lightgbm"}:
        raise ValueError("models must contain distinct choices from catboost/lightgbm")
    # Check requested dependencies before starting any expensive training.
    if "catboost" in models:
        from catboost import CatBoostClassifier
    if "lightgbm" in models:
        try:
            import lightgbm as lgb
        except ImportError as error:
            raise ImportError("Optional LightGBM is unavailable; install lightgbm or run CatBoost only.") from error

    project = Path(project)
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    assert train.drop(columns=TARGET).columns.tolist() == test.columns.tolist()
    assert train.id.is_unique and test.id.is_unique and not train.id.isin(test.id).any()
    assert not train.isna().any().any() and not test.isna().any().any()
    assert set(train[TARGET].unique()) == {"No", "Yes"}
    for frame in (train, test):
        assert np.allclose(frame.Annual_Income_USD, frame.Annual_Income_USD.round())
        assert np.allclose(frame.Daily_Commute_km * 10, (frame.Daily_Commute_km * 10).round())
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    out = project / "outputs" / ("v4_lightgbm_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    (out / "config.json").write_text(json.dumps({
        "models": models, "folds": 5, "seed": 42, "threads": threads,
        "train_rows": len(train), "test_rows": len(test),
        "note": "LightGBM/encoding and blend are unvalidated candidates; compare their OOF results.",
    }, indent=2))
    print("Output directory:", out, flush=True)
    all_oof, all_test, records = {}, {}, []

    def save_submission(name, probabilities):
        assert len(probabilities) == len(test)
        assert np.isfinite(probabilities).all()
        assert ((probabilities >= 0) & (probabilities <= 1)).all()
        pd.DataFrame({"id": test.id.to_numpy(), TARGET: probabilities}).to_csv(
            out / f"submission_{name}.csv", index=False)

    for name in models:
        x, xt = features(train, name), features(test, name)
        assert x.columns.tolist() == xt.columns.tolist()
        cats = x.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        oof = np.full(len(train), np.nan)
        coverage = np.zeros(len(train), dtype=np.int8)
        test_sum = np.zeros(len(test))
        for fold, (ti, vi) in enumerate(folds, start=1):
            print(f"\n========== {name} | Fold {fold}/5 ==========", flush=True)
            a, b, c = x.iloc[ti], x.iloc[vi], xt
            if name == "catboost":
                # Same features/parameters as the successful single-fold candidate.
                model = CatBoostClassifier(
                    iterations=2000, learning_rate=0.03, depth=7,
                    loss_function="Logloss", eval_metric="AUC", random_seed=41 + fold,
                    thread_count=threads, allow_writing_files=False, verbose=400,
                )
                model.fit(a, y.iloc[ti], cat_features=cats, eval_set=(b, y.iloc[vi]),
                          early_stopping_rounds=150, use_best_model=True)
                model.save_model(str(out / f"{name}_fold{fold}.cbm"))
                best = int(model.get_best_iteration()) + 1
            else:
                a, b, c = encode_fold(a, y.iloc[ti], b, c, seed=1041 + fold)
                bins = [1023 if col == "Annual_Income_USD" else 255 for col in a.columns]
                model = lgb.LGBMClassifier(
                    n_estimators=5000, learning_rate=0.05, max_depth=5, num_leaves=31,
                    min_child_samples=100, reg_lambda=5.0, max_bin_by_feature=bins,
                    objective="binary", metric="auc", random_state=41 + fold, n_jobs=threads,
                    verbosity=-1, deterministic=True, force_col_wise=True,
                )
                model.fit(a, y.iloc[ti], eval_set=[(b, y.iloc[vi])], eval_metric="auc",
                          callbacks=[lgb.early_stopping(200, first_metric_only=True), lgb.log_evaluation(400)])
                model.booster_.save_model(str(out / f"{name}_fold{fold}.txt"))
                best = int(model.best_iteration_)
                # Persist fold-specific TE maps' source indices, for reproducible later inference.
                np.save(out / f"{name}_fold{fold}_train_indices.npy", ti)
            assert np.array_equal(model.classes_, [0, 1])
            vp = model.predict_proba(b)[:, 1]
            tp = model.predict_proba(c)[:, 1]
            assert np.isfinite(vp).all() and np.isfinite(tp).all()
            oof[vi], coverage[vi] = vp, coverage[vi] + 1
            test_sum += tp
            auc = float(roc_auc_score(y.iloc[vi], vp))
            records.append({"model": name, "fold": fold, "auc": auc, "best_trees": best})
            np.savez_compressed(out / f"{name}_fold{fold}_predictions.npz",
                                valid_indices=vi, valid_ids=train.id.iloc[vi].to_numpy(),
                                valid_prediction=vp, test_ids=test.id.to_numpy(), test_prediction=tp)
            pd.DataFrame(records).to_csv(out / "fold_scores.csv", index=False)
            print("Fold AUC:", auc, flush=True)
        assert (coverage == 1).all() and np.isfinite(oof).all()
        all_oof[name], all_test[name] = oof, test_sum / len(folds)
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": oof}).to_csv(
            out / f"oof_{name}.csv", index=False)
        save_submission(name, all_test[name])
        print(name, "OOF AUC:", roc_auc_score(y, oof), flush=True)

    if len(models) == 2:
        # One predeclared blend, avoiding an extensive weight search on the same OOF.
        all_oof["blend_50_50"] = (all_oof["catboost"] + all_oof["lightgbm"]) / 2
        all_test["blend_50_50"] = (all_test["catboost"] + all_test["lightgbm"]) / 2
        save_submission("blend_50_50", all_test["blend_50_50"])
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(),
                      "prediction": all_oof["blend_50_50"]}).to_csv(out / "oof_blend_50_50.csv", index=False)
    scores = {name: float(roc_auc_score(y, prediction)) for name, prediction in all_oof.items()}
    best_name = max(scores, key=scores.get)
    save_submission("best_oof", all_test[best_name])
    (out / "summary.json").write_text(json.dumps({
        "oof_auc": scores, "selected": best_name,
        "selection_note": "Selected on development OOF; this is not an independent audit or a Kaggle score.",
    }, indent=2))
    print("\nOOF comparison:", scores)
    print("Selected:", best_name, "\nCandidate:", out / "submission_best_oof.csv")
    return out




## 2. 路径与旧结果核对工具

默认路径来自你的已保存日志。旧结果缺失、ID/标签不一致、折划分不同、CSV 不等于每折预测的汇总，都会先报错，不会悄悄混用结果。

这能核对旧运行的标签、预测和划分，但不能追溯证明旧原始特征从未被修改：V3 没有保存原 CSV 的哈希。请保留当时原始的 `train.csv` / `test.csv`；本版本会记录其哈希供后续复用核对。

In [2]:
import sys
import hashlib
import importlib.metadata


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def input_hashes(project):
    root = Path(project) / "data"
    names = ["train.csv", "test.csv", "sample_submission.csv"]
    return {name: file_hash(root / name) for name in names if (root / name).is_file()}


def probabilities(values, expected_length, description):
    p = np.asarray(values, dtype=float)
    require(p.shape == (expected_length,), f"{description}: wrong prediction shape")
    require(np.isfinite(p).all(), f"{description}: NaN or infinity")
    require(((p >= 0) & (p <= 1)).all(), f"{description}: probabilities outside [0, 1]")
    return p


def align_csv(path, ids, columns):
    """Do not blend CSVs by their incidental row order: align explicitly by id."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"缺少文件：{path}。请先确认原 outputs 目录；尚未开始新训练。")
    frame = pd.read_csv(path)
    require(set(["id", *columns]).issubset(frame.columns), f"{path.name}: missing columns")
    require(frame.id.notna().all() and frame.id.is_unique, f"{path.name}: invalid/duplicate ids")
    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted) and wanted.isin(frame.id).all(), f"{path.name}: id sets differ")
    return frame.set_index("id").loc[wanted, columns].reset_index()


def load_project(project):
    project = Path(project)
    for name in ("train.csv", "test.csv"):
        if not (project / "data" / name).is_file():
            raise FileNotFoundError(f"找不到 {project / 'data' / name}；请检查 PROJECT。")
    train = pd.read_csv(project / "data/train.csv")
    test = pd.read_csv(project / "data/test.csv")
    require(TARGET in train, "train.csv is missing the target")
    require(train.drop(columns=TARGET).columns.tolist() == test.columns.tolist(), "Train/test schema differs")
    require(train.id.is_unique and test.id.is_unique, "Duplicate id")
    require(not train.id.isin(test.id).any(), "Train/test ids overlap")
    require(not train.isna().any().any() and not test.isna().any().any(), "Missing values found")
    require(set(train[TARGET].unique()) == {"No", "Yes"}, "Expected No/Yes labels")
    y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
    require(y.value_counts().min() >= 10, "Too few rows per class for nested five-fold validation")
    folds = list(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
    sample = project / "data/sample_submission.csv"
    if sample.exists():
        sample_frame = pd.read_csv(sample)
        require(sample_frame.columns.tolist() == ["id", TARGET], "Unexpected submission columns")
        require(sample_frame.id.is_unique and len(sample_frame) == len(test), "Invalid sample submission ids")
        require(pd.Index(test.id).isin(sample_frame.id).all(), "Sample submission has different ids")
    return train, test, y, folds


def read_run(run_dir, model_name, train, test, y, folds, expected_auc=None):
    """Verify saved OOF, test averages, and the actual five validation splits."""
    run_dir = Path(run_dir)
    config_path = run_dir / "config.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"找不到 {config_path}；请把 BASELINE_DIR 指向原训练结果目录。")
    config = json.loads(config_path.read_text())
    require(config.get("folds") == 5 and config.get("seed") == 42, "Run does not use V3 folds/seed")
    require(model_name in config.get("models", []), f"This run does not contain {model_name}")
    require(config.get("train_rows") == len(train) and config.get("test_rows") == len(test), "Dataset sizes differ")
    oof_frame = align_csv(run_dir / f"oof_{model_name}.csv", train.id, ["target", "prediction"])
    test_frame = align_csv(run_dir / f"submission_{model_name}.csv", test.id, [TARGET])
    require(np.array_equal(oof_frame.target.to_numpy(), y.to_numpy()), "OOF labels do not match current train.csv")
    oof = probabilities(oof_frame.prediction, len(train), "OOF")
    test_pred = probabilities(test_frame[TARGET], len(test), "Test")
    reconstructed = np.full(len(train), np.nan)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test))
    fold_records = []
    for fold, (_, vi) in enumerate(folds, start=1):
        path = run_dir / f"{model_name}_fold{fold}_predictions.npz"
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；无法核对原五折，不会静默混用其他划分。")
        with np.load(path, allow_pickle=False) as saved:
            require(np.array_equal(saved["valid_indices"], vi), f"Fold {fold}: split differs")
            require(np.array_equal(saved["valid_ids"], train.id.iloc[vi]), f"Fold {fold}: train order/ids differ")
            require(np.array_equal(saved["test_ids"], test.id), f"Fold {fold}: test order/ids differ")
            vp = probabilities(saved["valid_prediction"], len(vi), f"Fold {fold} OOF")
            tp = probabilities(saved["test_prediction"], len(test), f"Fold {fold} test")
            reconstructed[vi] = vp
            coverage[vi] += 1
            test_sum += tp
            fold_records.append({"fold": fold, "auc": float(roc_auc_score(y.iloc[vi], vp))})
    require((coverage == 1).all(), "OOF coverage is not exactly once per row")
    require(np.allclose(reconstructed, oof, atol=1e-12, rtol=0), "OOF CSV differs from fold predictions")
    require(np.allclose(test_sum / 5, test_pred, atol=1e-12, rtol=0), "Test CSV is not this run's five-fold average")
    score = float(roc_auc_score(y, oof))
    if expected_auc is not None:
        require(abs(score - expected_auc) < 1e-8,
                f"旧 OOF AUC 是 {score:.10f}，不是本次预期的 {expected_auc:.10f}；请检查目录版本。")
    return {"oof": oof, "test": test_pred, "auc": score, "fold_scores": pd.DataFrame(fold_records)}


def compare_and_save(project, baseline_dir, lightgbm_dir, train, test, y, folds, old, new, hashes):
    """Four predeclared candidates only; this is development selection, not an audit."""
    require(input_hashes(project) == hashes, "Input CSV files changed during this session; stop and inspect")
    out = Path(project) / "outputs" / ("v4_compare_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    # Include the old baseline so a weaker LightGBM does not automatically replace it.
    candidates = [("catboost_baseline", 1.0), ("lightgbm", 0.0),
                  ("blend_cat75_lgb25", 0.75), ("blend_cat50_lgb50", 0.50)]
    records, per_fold, candidate_test = [], [], {}
    base_fold = [roc_auc_score(y.iloc[vi], old["oof"][vi]) for _, vi in folds]
    for name, weight in candidates:
        op = weight * old["oof"] + (1 - weight) * new["oof"]
        tp = probabilities(weight * old["test"] + (1 - weight) * new["test"], len(test), name)
        auc = float(roc_auc_score(y, op))
        deltas = []
        for fold, (_, vi) in enumerate(folds, start=1):
            fa = float(roc_auc_score(y.iloc[vi], op[vi]))
            delta = float(fa - base_fold[fold - 1])
            deltas.append(delta)
            per_fold.append({"candidate": name, "fold": fold, "auc": fa, "delta_vs_catboost": delta})
        records.append({"candidate": name, "catboost_weight": weight,
                        "oof_auc": auc, "delta_vs_catboost": auc - old["auc"],
                        "improved_folds": sum(d > 0 for d in deltas),
                        "worst_fold_delta": min(deltas)})
        pd.DataFrame({"id": train.id.to_numpy(), "target": y.to_numpy(), "prediction": op}).to_csv(
            out / f"oof_{name}.csv", index=False)
        submission = pd.DataFrame({"id": test.id.to_numpy(), TARGET: tp})
        sample_path = Path(project) / "data/sample_submission.csv"
        if sample_path.exists():
            sample_ids = pd.read_csv(sample_path, usecols=["id"])
            submission = sample_ids.merge(submission, on="id", how="left", validate="one_to_one", sort=False)
        probabilities(submission[TARGET], len(test), "Submission")
        submission.to_csv(out / f"submission_{name}.csv", index=False)
        candidate_test[name] = submission
    # max keeps the first candidate on an exact tie (the old baseline).
    best = max(records, key=lambda row: row["oof_auc"])
    comparison = pd.DataFrame(records).sort_values("oof_auc", ascending=False, kind="stable").reset_index(drop=True)
    comparison.to_csv(out / "comparison.csv", index=False)
    pd.DataFrame(per_fold).to_csv(out / "comparison_by_fold.csv", index=False)
    candidate_test[best["candidate"]].to_csv(out / "submission_next_candidate.csv", index=False)
    (out / "continuation_summary.json").write_text(json.dumps({
        "baseline_dir": str(baseline_dir), "lightgbm_dir": str(lightgbm_dir),
        "baseline_oof_auc": old["auc"], "user_reported_baseline_public_auc": 0.94570,
        "selected_by_development_oof": best, "candidates": records,
        "input_sha256": hashes,
        "note": "OOF was used for early stopping and model/weight selection; not an independent test. No new Kaggle score is known.",
    }, indent=2), encoding="utf-8")
    print(comparison.to_string(index=False, float_format=lambda v: f"{v:.8f}"))
    print("\n比较目录：", out)
    print("OOF 选择的候选：", best["candidate"])
    print("候选 CSV：", out / "submission_next_candidate.csv")
    if best["candidate"] == "catboost_baseline":
        print("本轮候选均未超过旧 CatBoost。保留原 0.94570 提交，不必为了新版本而重新提交。")
    else:
        print("这是开发 OOF 最优候选，不保证 Public/Private 分数更高。请同时检查逐折变化，不做大规模权重搜索。")
    return out, comparison


In [3]:
PROJECT = Path("/Users/yw/unsw/Kaggle")
BASELINE_DIR = PROJECT / "outputs/v3_20260925_180457_278867"
EXPECTED_BASELINE_AUC = 0.9456182325266642
THREADS = -1  # 保持原设定。需要同时使用电脑时可改成 4。

# 这份 notebook 已经完整跑过 LightGBM，因此默认直接复用已保存目录，避免 Run All 重训。
# 这里只复用【完整】结果，不支持某一折训练到一半的自动恢复。
LGB_RESULT_DIR = PROJECT / "outputs/v4_lightgbm_20260930_163526_079778"
# 示例（请用程序实际打印的目录，不要直接复制占位符）：
# LGB_RESULT_DIR = PROJECT / "outputs/v4_lightgbm_实际时间戳"


## 3. 先检查，不训练

正常情况下应打印旧 OOF AUC 约 `0.9456182325`。若报错缺少文件，请调整 `PROJECT` / `BASELINE_DIR` 或恢复旧输出，不要随便换成另一次训练目录。

`sample_submission.csv` 存在时会检查它并用其 ID 顺序导出；不存在时保留原 `test.csv` 的 ID 顺序。

In [4]:
# 这一格只检查，不训练。
try:
    import lightgbm as lgb
except (ImportError, OSError) as error:
    raise RuntimeError(
        "当前 Jupyter 内核无法导入 lightgbm。请先在这个内核安装/修复后重启内核；"
        "缺 Python 包时可在单独一格运行 %pip install lightgbm。"
    ) from error

train, test, y, folds = load_project(PROJECT)
old = read_run(BASELINE_DIR, "catboost", train, test, y, folds,
               expected_auc=EXPECTED_BASELINE_AUC)
DATA_HASHES = input_hashes(PROJECT)
print("当前 Python：", sys.executable)
print("旧结果检查通过；CatBoost OOF AUC：", old["auc"])
print(old["fold_scores"].to_string(index=False))
print("本轮不会重训 CatBoost。接下来运行第 4 节才会训练 LightGBM。")


当前 Python： /opt/miniconda3/envs/dl/bin/python
旧结果检查通过；CatBoost OOF AUC： 0.9456182325266642
 fold      auc
    1 0.944681
    2 0.945343
    3 0.946578
    4 0.945892
    5 0.945626
本轮不会重训 CatBoost。接下来运行第 4 节才会训练 LightGBM。


## 4. 只补跑 LightGBM 五折

参数保持原候选：5000 轮上限、学习率 0.05、深度 5、最多 31 个叶节点、连续 200 轮无提升早停。每折的最佳轮次由验证集决定。

第一次运行 `LGB_RESULT_DIR = None`。完整训练完成后程序会在同一内核记住目录；重启内核后，可以将第 2 节 `LGB_RESULT_DIR` 改成实际打印的目录，跳过训练。**中断在某一折时，原训练器没有自动断点续训；重新启动会新建目录并从五折的第一折重跑。**

不要因为这一节打印 `Selected: lightgbm` 就认为它比旧模型强：它在这一节是唯一参选模型，最终比较在下一节。

In [5]:
# 只有这里会启动训练；最多新训练一套 LightGBM 五折。
if LGB_RESULT_DIR is None:
    require(input_hashes(PROJECT) == DATA_HASHES, "检查后数据文件已改变")
    lightgbm_dir = run(project=PROJECT, models=("lightgbm",), threads=THREADS)
    require(input_hashes(PROJECT) == DATA_HASHES, "训练过程中数据文件已改变")
    (lightgbm_dir / "data_fingerprints.json").write_text(json.dumps(DATA_HASHES, indent=2))
    versions = {name: importlib.metadata.version(name)
                for name in ("numpy", "pandas", "scikit-learn", "lightgbm")}
    (lightgbm_dir / "package_versions.json").write_text(json.dumps(versions, indent=2))
    LGB_RESULT_DIR = lightgbm_dir  # 同一内核再次运行此格会复用完整结果。
else:
    lightgbm_dir = Path(LGB_RESULT_DIR)
    require(lightgbm_dir.resolve() != BASELINE_DIR.resolve(), "不能把旧 CatBoost 目录当成 LightGBM 目录")
    fingerprint_path = lightgbm_dir / "data_fingerprints.json"
    if not fingerprint_path.is_file():
        raise FileNotFoundError(f"缺少 {fingerprint_path}。请选择本 notebook 已完成的 LightGBM 目录。")
    require(json.loads(fingerprint_path.read_text()) == DATA_HASHES, "复用目录对应的数据文件版本不同")
    print("复用完整的 LightGBM 结果：", lightgbm_dir)

new = read_run(lightgbm_dir, "lightgbm", train, test, y, folds)
print("LightGBM OOF AUC：", new["auc"])
print("注意：此训练目录里的 submission_best_oof.csv 只在 LightGBM 本轮内选择。")
print("请继续运行第 5 节，才能与旧 CatBoost 一起比较。")


复用完整的 LightGBM 结果： /Users/yw/unsw/Kaggle/outputs/v4_lightgbm_20260930_163526_079778
LightGBM OOF AUC： 0.9459463810009869
注意：此训练目录里的 submission_best_oof.csv 只在 LightGBM 本轮内选择。
请继续运行第 5 节，才能与旧 CatBoost 一起比较。


## 5. 用相同 OOF 比较四个候选，导出提交文件

只比较预先规定的四个方案，不做密集权重搜索。75/25 表示 `0.75 × CatBoost + 0.25 × LightGBM`；50/50 同理。

先按 ID 对齐预测，然后计算整体 OOF AUC，并记录逐折相对旧 CatBoost 的变化。`improved_folds` 是五个验证折中有几个提高；它**不是显著性检验**。OOF 同时用于早停和选择，因此这些结果是开发分数，不是独立评估。

新目录 `v4_compare_时间戳` 中：`comparison.csv` 是总体比较，`comparison_by_fold.csv` 是逐折变化，`submission_next_candidate.csv` 是开发 OOF 选出的候选。旧模型也参与选择；新方案没超过旧模型时保留旧模型，不需要重复提交同一批预测。

In [6]:
comparison_dir, comparison = compare_and_save(
    PROJECT, BASELINE_DIR, lightgbm_dir, train, test, y, folds, old, new, DATA_HASHES
)
comparison


        candidate  catboost_weight    oof_auc  delta_vs_catboost  improved_folds  worst_fold_delta
blend_cat50_lgb50       0.50000000 0.94598624         0.00036801               5        0.00032103
         lightgbm       0.00000000 0.94594638         0.00032815               5        0.00024666
blend_cat75_lgb25       0.75000000 0.94585286         0.00023463               5        0.00020918
catboost_baseline       1.00000000 0.94561823         0.00000000               0        0.00000000

比较目录： /Users/yw/unsw/Kaggle/outputs/v4_compare_20260930_205345_105835
OOF 选择的候选： blend_cat50_lgb50
候选 CSV： /Users/yw/unsw/Kaggle/outputs/v4_compare_20260930_205345_105835/submission_next_candidate.csv
这是开发 OOF 最优候选，不保证 Public/Private 分数更高。请同时检查逐折变化，不做大规模权重搜索。


,candidate,catboost_weight,oof_auc,delta_vs_catboost,improved_folds,worst_fold_delta
0,blend_cat50_lgb50,0.50,0.945986,0.000368,5,0.000321
1,lightgbm,0.00,0.945946,0.000328,5,0.000247
2,blend_cat75_lgb25,0.75,0.945853,0.000235,5,0.000209
3,catboost_baseline,1.00,0.945618,0.000000,0,0.000000


## 怎么读结果、怎么决定下一步

以旧 OOF **0.9456182325** 为比较基线，而不是把它与 Public **0.94570** 跨数据集直接比输赢。即使新 OOF 更高，也不保证新的 Public 或 Private 更高。很小的提升尤其要看逐折变化；不要为了追逐 Public 的偶然变化反复调权重。

若 LightGBM 或融合没有提高，本轮仍然得到了有效的负结果：保持原方案。下一项可单独测试把 CatBoost 迭代上限从 2000 提到 3500，并保持特征、学习率和相同五折；这需要新训练，**本 notebook 没有启动该实验**。

参考：
- [竞赛评分与截止时间](https://www.kaggle.com/competitions/playground-series-s6e9/overview)
- [scikit-learn：ROC AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
- [scikit-learn：目标编码的内部交叉拟合](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_target_encoder_cross_val.html)
- [LightGBM：early_stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html)

算法主体延续用户上传的 V3 实现；新增部分负责旧运行核对、复用、ID 对齐、数据版本记录及四候选比较。

## 6. V5 参数：10-fold XGBoost

先跑这一节，不要急着改很多参数。

这次最重要的变化不是“换一组 XGB 参数”，而是：

1. **5-fold → 10-fold**；
2. 保留 digit decomposition；
3. 每个 outer fold 内做 frequency encoding；
4. 每个 outer fold 内做 **inner 5-fold 的 Triple Target Encoding**；
5. XGBoost 只看到训练折产生的编码，validation/test 不参与 target statistics；
6. 最后同时比较 raw probability blend 和 **rank blend**。

`XGB_RESULT_DIR = None` 时训练；完整训练结束后，把打印出来的目录填回去即可复用。


In [7]:
# ===== V5 configuration =====
CURRENT_PUBLIC_SCORE = 0.94618

XGB_SPLITS = 10
XGB_SEED = 42

# Triple TE：公开 S6E9 XGB 方案里常见的多平滑 target encoding。
TE_SMOOTHINGS = (1.0, 5.0, 20.0)
INNER_TE_SPLITS = 5

# 首次运行保持 None；跑完以后可改成实际目录以跳过重训。
XGB_RESULT_DIR = None
# 示例：
# XGB_RESULT_DIR = PROJECT / "outputs/v5_xgb10_20260930_xxxxxx"

# Apple Silicon / CPU 直接用 hist。不要在 Mac 上硬设 CUDA。
XGB_PARAMS = dict(
    n_estimators=4000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=20,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_lambda=8.0,
    reg_alpha=0.03,
    max_bin=512,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    early_stopping_rounds=200,
    n_jobs=THREADS,
)

print("Current reported Public:", CURRENT_PUBLIC_SCORE)
print("XGB folds:", XGB_SPLITS)
print("Triple-TE smoothings:", TE_SMOOTHINGS)


Current reported Public: 0.94618
XGB folds: 10
Triple-TE smoothings: (1.0, 5.0, 20.0)


## 7. XGBoost 特征：保留 exact-value / digit signal，但严格防 leakage

这一版特征分成三类：

- **row-local**：digit、income suffix/prefix、charging total/gap、income ratios、subsidy interactions；
- **frequency**：只使用当前 outer-training fold 的频数；
- **Triple TE**：outer-training fold 内再做 inner 5-fold OOF 编码。

特别注意：Target Encoding 不能先在完整 `train.csv` 上算好再切 10 folds。那样 CV 会虚高。


In [8]:
import gc

def add_xgb_row_features(frame):
    """Label-free row features. Safe to build before CV."""
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()

    digit_specs = {
        "Age": (1, 2),
        "Annual_Income_USD": (1, 6),
        "Daily_Commute_km": (10, 3),
        "Number_of_Cars_Owned": (1, 1),
        "Charging_Stations_Near_Home": (1, 2),
        "Charging_Stations_Near_Work": (1, 2),
        "Environmental_Concern_Level": (1, 1),
    }

    for col, (scale, positions) in digit_specs.items():
        if col not in frame.columns:
            continue
        values = (
            pd.to_numeric(frame[col], errors="coerce")
            .fillna(0)
            .to_numpy()
        )
        values = np.rint(values * scale).astype(np.int64)
        for digit in range(positions):
            x[f"{col}_digit_{digit}"] = (
                np.abs(values) // (10 ** digit) % 10
            ).astype(np.int8)

    if "Annual_Income_USD" in frame.columns:
        income = np.rint(
            pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").fillna(0)
        ).astype(np.int64)
        x["income_last_2"] = income % 100
        x["income_last_3"] = income % 1000
        x["income_prefix_100"] = income // 100
        x["income_prefix_1000"] = income // 1000

    if {"Charging_Stations_Near_Home", "Charging_Stations_Near_Work"} <= set(frame.columns):
        home = pd.to_numeric(frame["Charging_Stations_Near_Home"], errors="coerce").fillna(0)
        work = pd.to_numeric(frame["Charging_Stations_Near_Work"], errors="coerce").fillna(0)
        x["charging_total"] = home + work
        x["charging_gap"] = home - work

    if {"Annual_Income_USD", "Number_of_Cars_Owned"} <= set(frame.columns):
        income = pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").fillna(0)
        cars = pd.to_numeric(frame["Number_of_Cars_Owned"], errors="coerce").fillna(0)
        x["income_per_car"] = income / (cars + 1.0)

    if {"Annual_Income_USD", "Daily_Commute_km"} <= set(frame.columns):
        income = pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").fillna(0)
        commute = pd.to_numeric(frame["Daily_Commute_km"], errors="coerce").fillna(0)
        x["income_per_commute"] = income / (commute + 1.0)

    # Robust Yes/No -> 1/0 mapping for subsidy interactions.
    if "Subsidy_Available" in frame.columns:
        subsidy = frame["Subsidy_Available"]
        if pd.api.types.is_numeric_dtype(subsidy):
            subsidy_bin = pd.to_numeric(subsidy, errors="coerce")
        else:
            subsidy_bin = (
                subsidy.astype(str)
                .str.strip()
                .str.lower()
                .map({"yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0})
            )

        if subsidy_bin.notna().all():
            x["subsidy_bin"] = subsidy_bin.astype(np.int8).to_numpy()

            if "Annual_Income_USD" in frame.columns:
                income = pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").fillna(0)
                x["subsidy_x_income"] = subsidy_bin.to_numpy() * income.to_numpy()

            if "Environmental_Concern_Level" in frame.columns:
                concern = pd.to_numeric(
                    frame["Environmental_Concern_Level"], errors="coerce"
                ).fillna(0)
                x["subsidy_x_concern"] = subsidy_bin.to_numpy() * concern.to_numpy()

            if "Charging_Stations_Near_Home" in frame.columns:
                home = pd.to_numeric(
                    frame["Charging_Stations_Near_Home"], errors="coerce"
                ).fillna(0)
                x["subsidy_x_home_charging"] = subsidy_bin.to_numpy() * home.to_numpy()

    return x


def _is_cat(series):
    return (
        pd.api.types.is_object_dtype(series)
        or pd.api.types.is_string_dtype(series)
        or isinstance(series.dtype, pd.CategoricalDtype)
    )


def encode_xgb_fold(
    train_x,
    train_y,
    valid_x,
    test_x,
    seed,
    smoothings=TE_SMOOTHINGS,
    inner_splits=INNER_TE_SPLITS,
):
    """Fold-local frequency + inner-OOF multi-scale target encoding."""
    a, b, c = [d.copy().reset_index(drop=True) for d in (train_x, valid_x, test_x)]
    labels = np.asarray(train_y, dtype=np.int8)

    original_cols = train_x.columns.tolist()
    cat_cols = [col for col in original_cols if _is_cat(a[col])]
    numeric_exact = [col for col in NUMERIC if col in a.columns]

    # Avoid duplicate keys while keeping deterministic order.
    te_keys = list(dict.fromkeys(cat_cols + numeric_exact))

    inner = list(
        StratifiedKFold(
            inner_splits,
            shuffle=True,
            random_state=seed,
        ).split(a, labels)
    )

    for col in te_keys:
        for smoothing in smoothings:
            encoded = np.full(len(a), np.nan, dtype=np.float32)

            for fit_idx, hold_idx in inner:
                values = a[col].iloc[fit_idx].to_numpy()
                yy = labels[fit_idx]
                prior = float(yy.mean())

                stats = (
                    pd.DataFrame({"key": values, "y": yy})
                    .groupby("key", dropna=False)["y"]
                    .agg(["sum", "count"])
                )
                mapping = (
                    stats["sum"] + smoothing * prior
                ) / (
                    stats["count"] + smoothing
                )

                encoded[hold_idx] = (
                    a[col].iloc[hold_idx]
                    .map(mapping)
                    .fillna(prior)
                    .to_numpy(dtype=np.float32)
                )

            prior = float(labels.mean())
            stats = (
                pd.DataFrame({"key": a[col].to_numpy(), "y": labels})
                .groupby("key", dropna=False)["y"]
                .agg(["sum", "count"])
            )
            mapping = (
                stats["sum"] + smoothing * prior
            ) / (
                stats["count"] + smoothing
            )

            tag = str(smoothing).replace(".", "p")
            a[f"{col}_te_{tag}"] = encoded
            b[f"{col}_te_{tag}"] = (
                b[col].map(mapping).fillna(prior).to_numpy(dtype=np.float32)
            )
            c[f"{col}_te_{tag}"] = (
                c[col].map(mapping).fillna(prior).to_numpy(dtype=np.float32)
            )

            require(np.isfinite(encoded).all(), f"{col} TE contains non-finite values")

    # Label-free frequency encoding from this outer training fold only.
    n = len(a)
    for col in te_keys:
        counts = a[col].value_counts(dropna=False)
        a[f"{col}_freq"] = (
            (a[col].map(counts) - 1).clip(lower=0) / max(n - 1, 1)
        ).astype(np.float32)
        b[f"{col}_freq"] = (
            b[col].map(counts).fillna(0) / n
        ).astype(np.float32)
        c[f"{col}_freq"] = (
            c[col].map(counts).fillna(0) / n
        ).astype(np.float32)

    # XGBoost needs numeric input here. Encode raw categoricals using
    # categories learned ONLY from the outer training fold.
    for col in cat_cols:
        train_values = a[col].astype(str)
        categories = pd.Index(pd.unique(train_values))
        mapping = {value: idx for idx, value in enumerate(categories)}

        for frame in (a, b, c):
            frame[col] = (
                frame[col].astype(str)
                .map(mapping)
                .fillna(-1)
                .astype(np.int32)
            )

    for frame in (a, b, c):
        for col in frame.columns:
            if pd.api.types.is_bool_dtype(frame[col]):
                frame[col] = frame[col].astype(np.int8)

        bad = frame.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        require(not bad, f"Non-numeric XGB columns remain: {bad}")

        require(np.isfinite(frame.to_numpy(dtype=np.float64, copy=False)).all(),
                "Non-finite values in XGB feature matrix")

    require(a.columns.tolist() == b.columns.tolist() == c.columns.tolist(),
            "XGB train/valid/test feature columns differ")

    return a, b, c


## 8. 训练 10-fold XGBoost

这一步是 V5 的主要训练。

预期现象不是“每一折都一定超过你原来的 5-fold 模型”，而是：

- overall XGB OOF 尽量接近或超过 `0.9460`；
- 更理想是进入 `0.9462+`；
- 如果能靠近 heuljax 公布的 `0.9465` CV 区域，再考虑小幅调参；
- 如果 XGB OOF 本身一般，但与 Cat/LGB 的 OOF rank correlation 下降，仍可能通过 blend 提升。

训练完会保存每折模型、OOF、test prediction 和配置。


In [9]:
def run_xgb10(project, result_dir=None):
    try:
        import xgboost as xgb
    except ImportError as error:
        raise RuntimeError(
            "当前内核缺少 xgboost。请在单独一格运行 %pip install xgboost，"
            "安装后重启 kernel。"
        ) from error

    train, test, y, _ = load_project(project)
    hashes_before = input_hashes(project)

    x_all = add_xgb_row_features(train)
    x_test = add_xgb_row_features(test)
    require(x_all.columns.tolist() == x_test.columns.tolist(),
            "Row-local XGB train/test features differ")

    folds10 = list(
        StratifiedKFold(
            XGB_SPLITS,
            shuffle=True,
            random_state=XGB_SEED,
        ).split(train, y)
    )

    if result_dir is not None:
        out = Path(result_dir)
        config_path = out / "config.json"
        if not config_path.is_file():
            raise FileNotFoundError(f"Missing {config_path}")

        config = json.loads(config_path.read_text())
        require(config.get("folds") == XGB_SPLITS, "Saved XGB fold count differs")
        require(config.get("seed") == XGB_SEED, "Saved XGB seed differs")
        require(config.get("input_sha256") == hashes_before, "Saved XGB data hashes differ")

        oof_frame = align_csv(out / "oof_xgboost.csv", train.id, ["target", "prediction"])
        test_frame = align_csv(out / "submission_xgboost.csv", test.id, [TARGET])

        require(np.array_equal(oof_frame["target"].to_numpy(), y.to_numpy()),
                "Saved XGB OOF labels differ from current train.csv")

        oof = probabilities(oof_frame["prediction"], len(train), "XGB OOF")
        test_pred = probabilities(test_frame[TARGET], len(test), "XGB test")
        score = float(roc_auc_score(y, oof))

        print("复用 XGBoost：", out)
        print("XGB OOF AUC:", score)
        return {
            "dir": out,
            "oof": oof,
            "test": test_pred,
            "auc": score,
            "folds": folds10,
        }

    out = (
        Path(project)
        / "outputs"
        / ("v5_xgb10_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    )
    out.mkdir(parents=True, exist_ok=False)

    config = {
        "model": "xgboost",
        "folds": XGB_SPLITS,
        "seed": XGB_SEED,
        "inner_te_splits": INNER_TE_SPLITS,
        "te_smoothings": list(TE_SMOOTHINGS),
        "xgb_params": XGB_PARAMS,
        "train_rows": len(train),
        "test_rows": len(test),
        "input_sha256": hashes_before,
        "note": (
            "10-fold XGB inspired by public S6E9 discussion. "
            "All target encoding is nested inside outer training folds."
        ),
    }
    (out / "config.json").write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    oof = np.full(len(train), np.nan, dtype=np.float64)
    coverage = np.zeros(len(train), dtype=np.int8)
    test_sum = np.zeros(len(test), dtype=np.float64)
    fold_rows = []

    for fold, (ti, vi) in enumerate(folds10, start=1):
        print(f"\\n========== XGBoost | Fold {fold}/{XGB_SPLITS} ==========", flush=True)

        a0 = x_all.iloc[ti]
        b0 = x_all.iloc[vi]
        c0 = x_test

        a, b, c = encode_xgb_fold(
            a0,
            y.iloc[ti],
            b0,
            c0,
            seed=3000 + fold,
        )

        params = dict(XGB_PARAMS)
        params["random_state"] = 1000 + fold

        model = xgb.XGBClassifier(**params)
        model.fit(
            a,
            y.iloc[ti],
            eval_set=[(b, y.iloc[vi])],
            verbose=200,
        )

        vp = model.predict_proba(b)[:, 1]
        tp = model.predict_proba(c)[:, 1]

        require(np.isfinite(vp).all() and np.isfinite(tp).all(),
                f"Fold {fold}: non-finite predictions")

        oof[vi] = vp
        coverage[vi] += 1
        test_sum += tp

        fold_auc = float(roc_auc_score(y.iloc[vi], vp))
        best_iteration = getattr(model, "best_iteration", None)

        fold_rows.append({
            "fold": fold,
            "auc": fold_auc,
            "best_iteration": None if best_iteration is None else int(best_iteration),
            "n_features": int(a.shape[1]),
        })

        model.save_model(str(out / f"xgboost_fold{fold}.json"))

        np.savez_compressed(
            out / f"xgboost_fold{fold}_predictions.npz",
            valid_indices=vi,
            valid_ids=train.id.iloc[vi].to_numpy(),
            valid_prediction=vp,
            test_ids=test.id.to_numpy(),
            test_prediction=tp,
        )

        pd.DataFrame(fold_rows).to_csv(out / "fold_scores.csv", index=False)

        print(f"Fold {fold} AUC: {fold_auc:.9f}")
        print("Best iteration:", best_iteration)

        del a0, b0, c0, a, b, c, model, vp, tp
        gc.collect()

    require((coverage == 1).all(), "XGB OOF coverage is not exactly one")
    require(np.isfinite(oof).all(), "XGB OOF has missing values")
    require(input_hashes(project) == hashes_before, "Input files changed during XGB training")

    test_pred = test_sum / XGB_SPLITS
    total_auc = float(roc_auc_score(y, oof))

    pd.DataFrame({
        "id": train.id.to_numpy(),
        "target": y.to_numpy(),
        "prediction": oof,
    }).to_csv(out / "oof_xgboost.csv", index=False)

    submission = pd.DataFrame({
        "id": test.id.to_numpy(),
        TARGET: test_pred,
    })
    sample_path = Path(project) / "data/sample_submission.csv"
    if sample_path.exists():
        sample_ids = pd.read_csv(sample_path, usecols=["id"])
        submission = sample_ids.merge(
            submission,
            on="id",
            how="left",
            validate="one_to_one",
            sort=False,
        )

    probabilities(submission[TARGET], len(test), "XGB submission")
    submission.to_csv(out / "submission_xgboost.csv", index=False)

    (out / "summary.json").write_text(
        json.dumps({
            "oof_auc": total_auc,
            "fold_scores": fold_rows,
            "reported_public_before_v5": CURRENT_PUBLIC_SCORE,
        }, indent=2),
        encoding="utf-8",
    )

    print("\\nXGBoost 10-fold OOF AUC:", total_auc)
    print("Result directory:", out)
    print("Single-model submission:", out / "submission_xgboost.csv")

    return {
        "dir": out,
        "oof": oof,
        "test": test_pred,
        "auc": total_auc,
        "folds": folds10,
    }


if XGB_RESULT_DIR is None:
    xgb_result = run_xgb10(PROJECT)
    XGB_RESULT_DIR = xgb_result["dir"]
else:
    xgb_result = run_xgb10(PROJECT, XGB_RESULT_DIR)


\n========== XGBoost | Fold 1/10 ==========
[0]	validation_0-auc:0.93711
[200]	validation_0-auc:0.94439
[400]	validation_0-auc:0.94477
[600]	validation_0-auc:0.94490
[800]	validation_0-auc:0.94493
[1000]	validation_0-auc:0.94492
[1053]	validation_0-auc:0.94491
Fold 1 AUC: 0.944937167
Best iteration: 853
\n========== XGBoost | Fold 2/10 ==========
[0]	validation_0-auc:0.93649
[200]	validation_0-auc:0.94443
[400]	validation_0-auc:0.94495
[600]	validation_0-auc:0.94507
[800]	validation_0-auc:0.94508
[900]	validation_0-auc:0.94508
Fold 2 AUC: 0.945096138
Best iteration: 700
\n========== XGBoost | Fold 3/10 ==========
[0]	validation_0-auc:0.93799
[200]	validation_0-auc:0.94486
[400]	validation_0-auc:0.94526
[600]	validation_0-auc:0.94536
[800]	validation_0-auc:0.94536
[850]	validation_0-auc:0.94536
Fold 3 AUC: 0.945372989
Best iteration: 650
\n========== XGBoost | Fold 4/10 ==========
[0]	validation_0-auc:0.93851
[200]	validation_0-auc:0.94527
[400]	validation_0-auc:0.94562
[600]	validation

## 9. 和你当前 0.94618 路线做概率融合 + Rank Blend

AUC 只关心排序，因此当不同模型的概率尺度不一致时，rank blend 往往比直接平均概率更稳定。

这里**不做上百个权重的搜索**，只预先比较少量候选：

- 你当前 CatBoost/LGB 50/50；
- XGBoost 单模型；
- 70% XGB + 30% 当前模型（probability）；
- 70% XGB + 30% 当前模型（rank）；
- 50% XGB + 50% 当前模型（rank）。

同时输出 XGB 与当前模型的 Spearman rank correlation。相关性越低，XGB 越可能提供真正的新信息。


In [10]:
def rank01(values):
    values = np.asarray(values, dtype=np.float64)
    return pd.Series(values).rank(method="average", pct=True).to_numpy(dtype=np.float64)


def compare_v5_xgb_blends(
    project,
    train,
    test,
    y,
    old,
    new,
    xgb_result,
):
    # Current best development route from the uploaded notebook.
    current_oof = 0.5 * old["oof"] + 0.5 * new["oof"]
    current_test = 0.5 * old["test"] + 0.5 * new["test"]

    xgb_oof = xgb_result["oof"]
    xgb_test = xgb_result["test"]

    current_rank_oof = rank01(current_oof)
    xgb_rank_oof = rank01(xgb_oof)
    current_rank_test = rank01(current_test)
    xgb_rank_test = rank01(xgb_test)

    spearman_corr = float(
        pd.Series(current_oof).corr(pd.Series(xgb_oof), method="spearman")
    )
    pearson_corr = float(np.corrcoef(current_oof, xgb_oof)[0, 1])

    candidates = {
        "current_cat50_lgb50": (
            current_oof,
            current_test,
        ),
        "xgb_single": (
            xgb_oof,
            xgb_test,
        ),
        "xgb70_current30_prob": (
            0.70 * xgb_oof + 0.30 * current_oof,
            0.70 * xgb_test + 0.30 * current_test,
        ),
        "xgb70_current30_rank": (
            0.70 * xgb_rank_oof + 0.30 * current_rank_oof,
            0.70 * xgb_rank_test + 0.30 * current_rank_test,
        ),
        "xgb50_current50_rank": (
            0.50 * xgb_rank_oof + 0.50 * current_rank_oof,
            0.50 * xgb_rank_test + 0.50 * current_rank_test,
        ),
    }

    out = (
        Path(project)
        / "outputs"
        / ("v5_compare_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    )
    out.mkdir(parents=True, exist_ok=False)

    rows = []
    per_fold = []
    folds10 = xgb_result["folds"]

    current_auc = float(roc_auc_score(y, current_oof))
    xgb_auc = float(roc_auc_score(y, xgb_oof))

    for name, (oof_pred, test_pred) in candidates.items():
        auc = float(roc_auc_score(y, oof_pred))

        fold_deltas_vs_current = []
        for fold, (_, vi) in enumerate(folds10, start=1):
            candidate_fold_auc = float(roc_auc_score(y.iloc[vi], oof_pred[vi]))
            current_fold_auc = float(roc_auc_score(y.iloc[vi], current_oof[vi]))
            delta = candidate_fold_auc - current_fold_auc

            fold_deltas_vs_current.append(delta)
            per_fold.append({
                "candidate": name,
                "fold": fold,
                "auc": candidate_fold_auc,
                "delta_vs_current": delta,
            })

        rows.append({
            "candidate": name,
            "oof_auc": auc,
            "delta_vs_current": auc - current_auc,
            "delta_vs_xgb": auc - xgb_auc,
            "improved_xgb_folds_vs_current": int(
                sum(d > 0 for d in fold_deltas_vs_current)
            ),
            "worst_fold_delta_vs_current": float(min(fold_deltas_vs_current)),
        })

        pd.DataFrame({
            "id": train.id.to_numpy(),
            "target": y.to_numpy(),
            "prediction": oof_pred,
        }).to_csv(out / f"oof_{name}.csv", index=False)

        sub = pd.DataFrame({
            "id": test.id.to_numpy(),
            TARGET: probabilities(test_pred, len(test), name),
        })

        sample_path = Path(project) / "data/sample_submission.csv"
        if sample_path.exists():
            sample_ids = pd.read_csv(sample_path, usecols=["id"])
            sub = sample_ids.merge(
                sub,
                on="id",
                how="left",
                validate="one_to_one",
                sort=False,
            )

        probabilities(sub[TARGET], len(test), f"{name} submission")
        sub.to_csv(out / f"submission_{name}.csv", index=False)

    table = (
        pd.DataFrame(rows)
        .sort_values("oof_auc", ascending=False, kind="stable")
        .reset_index(drop=True)
    )
    table.to_csv(out / "comparison.csv", index=False)
    pd.DataFrame(per_fold).to_csv(out / "comparison_by_fold.csv", index=False)

    best_name = table.iloc[0]["candidate"]
    best_submission = pd.read_csv(out / f"submission_{best_name}.csv")
    best_submission.to_csv(out / "submission_next_candidate.csv", index=False)

    diagnostics = {
        "current_oof_auc": current_auc,
        "xgb_oof_auc": xgb_auc,
        "reported_current_public": CURRENT_PUBLIC_SCORE,
        "spearman_xgb_vs_current": spearman_corr,
        "pearson_xgb_vs_current": pearson_corr,
        "selected_by_oof": best_name,
    }
    (out / "diagnostics.json").write_text(
        json.dumps(diagnostics, indent=2),
        encoding="utf-8",
    )

    print("Current Cat/LGB OOF:", current_auc)
    print("XGB OOF:", xgb_auc)
    print("Spearman(XGB, current):", spearman_corr)
    print("Pearson(XGB, current):", pearson_corr)
    print()
    print(table.to_string(index=False, float_format=lambda v: f"{v:.9f}"))
    print("\\nComparison directory:", out)
    print("OOF-selected candidate:", best_name)
    print("Candidate submission:", out / "submission_next_candidate.csv")
    print("\\n也保留了 XGB 单模型文件：", xgb_result["dir"] / "submission_xgboost.csv")

    return out, table, diagnostics


# Re-load/verify the saved 5-fold models from your existing V4 state.
train, test, y, folds = load_project(PROJECT)
old = read_run(
    BASELINE_DIR,
    "catboost",
    train,
    test,
    y,
    folds,
    expected_auc=EXPECTED_BASELINE_AUC,
)
new = read_run(
    Path(LGB_RESULT_DIR),
    "lightgbm",
    train,
    test,
    y,
    folds,
)

v5_compare_dir, v5_comparison, v5_diagnostics = compare_v5_xgb_blends(
    PROJECT,
    train,
    test,
    y,
    old,
    new,
    xgb_result,
)

v5_comparison


Current Cat/LGB OOF: 0.945986243390633
XGB OOF: 0.9459315426788591
Spearman(XGB, current): 0.9963674136703535
Pearson(XGB, current): 0.9979214260945515

           candidate     oof_auc  delta_vs_current  delta_vs_xgb  improved_xgb_folds_vs_current  worst_fold_delta_vs_current
xgb50_current50_rank 0.946098745       0.000112502   0.000167203                             10                  0.000061974
xgb70_current30_rank 0.946064667       0.000078424   0.000133124                             10                  0.000014073
xgb70_current30_prob 0.946060821       0.000074577   0.000129278                             10                  0.000002141
 current_cat50_lgb50 0.945986243       0.000000000   0.000054701                              0                  0.000000000
          xgb_single 0.945931543      -0.000054701   0.000000000                              2                 -0.000139096
\nComparison directory: /Users/yw/unsw/Kaggle/outputs/v5_compare_20260930_205844_305144
OOF-selec

,candidate,oof_auc,delta_vs_current,delta_vs_xgb,improved_xgb_folds_vs_current,worst_fold_delta_vs_current
0,xgb50_current50_rank,0.946099,0.000113,0.000167,10,0.000062
1,xgb70_current30_rank,0.946065,0.000078,0.000133,10,0.000014
2,xgb70_current30_prob,0.946061,0.000075,0.000129,10,0.000002
3,current_cat50_lgb50,0.945986,0.000000,0.000055,0,0.000000
4,xgb_single,0.945932,-0.000055,0.000000,2,-0.000139


## 10. 怎么决定提交哪个

优先看 `comparison.csv` 和 10 个 fold 的一致性，不要只看一个总 OOF 数字。

建议顺序：

1. **先看 `xgb_single` 的 OOF**。如果它已经明显高于 `0.945986`，这次方向就是有效的。
2. 再看 `xgb70_current30_rank`。如果 OOF 更高，并且多数 fold 都比当前 Cat/LGB 好，它比盲目 50/50 更值得提交。
3. 如果两个 blend 的差只有 `1e-5 ~ 3e-5`，不要因为这一点点差距继续搜几十个权重。
4. `Spearman(XGB, current)` 如果明显低于 1，说明 XGB 提供了新的排序信息；如果接近 0.9999，blend 空间会很小。
5. Public LB 是 20% 左右的测试子集表现，不应反过来用它暴力搜索 blend 权重。

### 下一步（先不要同时做）
heuljax 还公开提到 Logistic Regression CV 约 `0.94640`。如果这版 XGB 跑完后仍想继续，我们下一轮再加 **exact-value one-hot Logistic Regression**，作为和树模型结构不同的第三个模型。它的价值更可能来自“模型多样性”，不是单纯替换 XGB。

### 本文件的验证状态
- 原 V4 输出保留；
- V5 新代码已做语法检查；
- 特征/TE/XGB 训练函数已用合成数据做过功能测试；
- **没有你的 `train.csv/test.csv`，因此这里无法替你完成真实 10-fold 训练**。请在你原 `/Users/yw/unsw/Kaggle` 环境中运行第 6–9 节。
